**22 | Four Advanced Temporal Architectures: Implementation and Engineering Validation**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research objective:** Implement and verify four genuinely distinct neural architectures for six-channel HARTH IMU classification: Temporal Convolutional Network (TCN), Multi-Scale CNN (InceptionTime-inspired), 1D-ResNet, and Time-Series Transformer. This notebook explains their computational designs, checks architecture contracts, calculates parameter counts, and runs a small **real-source engineering smoke test**. Stage 23, not this notebook, conducts the matched subject-held-out scientific performance benchmark.

**Data boundary:** The original S023 and S032 reserved model-development Test participants are never loaded. Training examples in the smoke test come from a deliberately small, independent-participant development subset; absence of some class labels is expected and prevents drawing meaningful comparative F1 conclusions.

**01. Study Design and FAIR Architecture Comparison Rules**

Each model accepts precisely the same tensor format, `(batch, 6, 100)`, and returns eight raw class logits, `(batch, 8)`. Labels remain official documented source codes `1`–`8`; any class-index encoding is confined to the training loss and must be reversible. Cross-Entropy receives logits, never pre-applied Softmax. To establish fair evaluation in Stage 23, we will hold constant source-window identity, Train/Validation participant groups, source class specification, preprocessing policy and primary Macro F1 definition, while reporting differences in model complexity, actual elapsed training time and compute. Equal optimiser steps do not imply equal computational cost or equivalent inductive bias.

In [ ]:
# ==================== 01. Imports, Research Setup and Common Inputs ====================

# 1. Locate repository and import tested modules without creating an HMI package.
import sys
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display
root=Path.cwd().resolve()
while not (root/"code/python/src/core/data_loading.py").is_file():
    if root==root.parent:
        raise RuntimeError("Open this notebook within the repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
sys.path.insert(0,str(root/"studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS, prepare_harth_window_cohort
from core.advanced_temporal_models import (
    ARCHITECTURES, build_temporal_model, count_trainable_parameters
)
from core.temporal_cnn import TemporalCNN, standardise_imu_train_only

# 2. Fix the reproducibility seed and recorded hardware environment.
SEED=42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
CLASSES=tuple(range(1,9))
RESERVED_TEST={"S023","S032"}
assert len(SENSOR_COLUMNS)==6 and len(CLASSES)==8
print("\n"+"="*66)
print("STAGE 22: FOUR ADVANCED TEMPORAL ARCHITECTURES")
print("="*66)
print("PyTorch:",torch.__version__)
print("Device:",DEVICE)
print("New architectures:",", ".join(ARCHITECTURES))
print("Reserved Test excluded:",sorted(RESERVED_TEST))
print("="*66)

**02. Architecture A: Temporal Convolutional Network (TCN)**

The TCN uses stacked residual blocks with exponentially increasing dilations (`1, 2, 4, 8`). This enlarges the temporal receptive field without requiring aggressive early downsampling. Each block includes two five-point dilated convolutions, BatchNorm, GELU, Dropout and residual projection when the channel width changes. Our chosen formulation is **noncausal**, appropriate for classifying an already-completed two-second window; it should not be presented as a streaming forecaster. The model ends in global temporal pooling and a class-logit layer.

In [ ]:
# ==================== 02. TCN Forward and Receptive-Field Inspection ====================

# 1. Construct the declared noncausal TCN on a fixed source-style input.
tcn=build_temporal_model("TCN",in_channels=6,n_classes=8)
sample=torch.randn(3,6,100)
with torch.no_grad():
    tcn.eval()
    tcn_logits=tcn(sample)

# 2. Verify the exact common output contract and report model capacity.
assert tcn_logits.shape==(3,8)
print("TCN input:",tuple(sample.shape))
print("TCN logits:",tuple(tcn_logits.shape))
print("TCN trainable parameters:",f"{count_trainable_parameters(tcn):,}")
print("Temporal dilations:",[1,2,4,8])
print("Noncausal full-window classification: YES")

**03. Architecture B: Multi-Scale CNN (InceptionTime-Inspired)**

A single temporal kernel size may fail to capture patterns at several resolutions. This model concatenates three convolution branches (`kernel_size=3, 7, 15`) and an additional MaxPooling branch, after an input bottleneck. Residual shortcuts and BatchNorm are used before global average pooling. This is a **small InceptionTime-inspired architecture**, not the complete published InceptionTime ensemble; avoid mislabelling it as an exact implementation of a reference model.

In [ ]:
# ==================== 03. Parallel-Kernel Multi-Scale CNN Inspection ====================

# 1. Initialise the model and verify shape and class-logit semantics.
multi=build_temporal_model("Multi-Scale CNN",in_channels=6,n_classes=8)
multi.eval()
with torch.no_grad():
    multiscale_logits=multi(sample)
assert multiscale_logits.shape==(3,8)

# 2. Document the actual temporal feature extraction resolutions.
print("Multi-Scale CNN logits:",tuple(multiscale_logits.shape))
print("Parallel kernel sizes:",[3,7,15])
print("Additional pooling branch: YES")
print("Trainable parameters:",f"{count_trainable_parameters(multi):,}")
print("Full original InceptionTime ensemble implemented: NO")

**04. Architecture C: 1D-ResNet**

ResNet1D combines an initial temporal convolution with three residual stages. The second stage doubles feature channels and reduces temporal resolution with stride two; identity or projected shortcuts keep feature dimensions compatible. Global pooling produces a fixed-length representation. Skip connections ease gradient flow in deeper stacks but do not guarantee greater accuracy; this model is a lightweight time-series network, not a 2D image ResNet.

In [ ]:
# ==================== 04. One-Dimensional Residual Network Inspection ====================

# 1. Confirm output shape and the use of residual blocks.
resnet=build_temporal_model("1D-ResNet",in_channels=6,n_classes=8)
resnet.eval()
with torch.no_grad():
    resnet_logits=resnet(sample)

# 2. Record network size without asserting it is superior.
assert resnet_logits.shape==(3,8)
print("1D-ResNet logits:",tuple(resnet_logits.shape))
print("Trainable parameters:",f"{count_trainable_parameters(resnet):,}")
print("Strided temporal downsampling: YES")
print("Full-window global pooling: YES")

**05. Architecture D: Time-Series Transformer**

A fully trainable input projection maps each six-channel time point to a 48-dimensional embedding. Fixed sinusoidal positional encodings are added to preserve sample order. Two Transformer encoder layers with four attention heads model bidirectional token interactions across the 100-sample window. LayerNorm, average token pooling and a final classification head create eight logits. Attention is **bidirectional**, not streaming causal; on a small grouped dataset the model may overfit or be computationally costly. Attention weights are not claimed as faithful explanations.

In [ ]:
# ==================== 05. Time-Series Self-Attention Inspection ====================

# 1. Build the deliberately compact attention architecture.
transformer=build_temporal_model(
    "Time-Series Transformer",in_channels=6,n_classes=8,
    d_model=48,n_heads=4,n_layers=2,feedforward_dim=96,max_length=256
)
transformer.eval()
with torch.no_grad():
    transformer_logits=transformer(sample)

# 2. Check position encoding, output contract and parameter cost.
assert transformer_logits.shape==(3,8)
print("Transformer logits:",tuple(transformer_logits.shape))
print("Attention heads:",4,"| Encoder layers:",2)
print("Embedding dimension:",48)
print("Trainable parameters:",f"{count_trainable_parameters(transformer):,}")
print("Causal real-time behaviour established: NO")

**06. Shared Architecture Contract, Sizes and CPU Inference Timing**

Raw parameter counts and a small forward-pass timing probe can indicate engineering cost. They are **not** calibrated production latency benchmarks: CPU/GPU hardware, thread scheduling, warm-up, input size and PyTorch version change measured timings. Every model is set to evaluation mode and receives the same float32 synthetic input solely for this engineering check. No labels are involved.

In [ ]:
# ==================== 06. Unified Capacity and Forward-Latency Table ====================

# 1. Include the existing 1D-CNN as an architectural reference.
reference={"Simple Temporal CNN":TemporalCNN(6,8)}
models={name:build_temporal_model(name,6,8) for name in ARCHITECTURES}
models={**reference,**models}
timing_input=torch.randn(16,6,100)
rows=[]

# 2. Use the same tiny warm-up and measured inference batches on CPU.
for name,model in models.items():
    model=model.cpu().eval()
    with torch.no_grad():
        for _ in range(2):
            logits=model(timing_input)
        starts=time.perf_counter()
        for _ in range(5):
            logits=model(timing_input)
        elapsed=(time.perf_counter()-starts)/5*1000
    assert logits.shape==(16,8) and torch.isfinite(logits).all()
    rows.append({
        "architecture":name,
        "trainable_parameters":count_trainable_parameters(model),
        "cpu_ms_per_16_windows":elapsed,
        "logits_shape":str(tuple(logits.shape))
    })
capacity=pd.DataFrame(rows)
display(capacity.round(2))

# 3. Visualise architectural complexity without implying predictive superiority.
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].barh(capacity["architecture"],capacity["trainable_parameters"])
axes[0].set(title="Trainable architecture parameters",xlabel="Parameters")
axes[1].barh(capacity["architecture"],capacity["cpu_ms_per_16_windows"])
axes[1].set(title="Local CPU inference timing (illustrative)",xlabel="Milliseconds per 16 windows")
for ax in axes:
    ax.grid(axis="x",alpha=0.2)
fig.tight_layout()
plt.show()

**07. Real HARTH Development Source and Participant Boundary**

The engineering smoke test uses a small sample of actual HARTH v2.0 windows from four development participants only: S006, S008, S012 and S013. We build legitimate two-second, pure-label and timestamp-continuous windows with the existing adapter. Train and validation are separated by **participant identity**, not by individual overlapping window row. The reserved original Test participants are excluded. This intentionally small subset cannot support valid conclusions about rare-class recall or generalisation.

In [ ]:
# ==================== 07. Genuine HARTH Window and Group Audit ====================

# 1. Load four genuine development subjects with only 32 eligible windows each.
SMOKE_PEOPLE=["S006","S008","S012","S013"]
assert not(set(SMOKE_PEOPLE)&RESERVED_TEST)
X,y,metadata,overview=prepare_harth_window_cohort(
    subjects=SMOKE_PEOPLE,data_dir=root/"data/raw/harth_v2",
    task_classes=CLASSES,max_windows_per_subject=32,
    window_size=100,step_size=50,sampling_rate=50
)
display(overview[["subject_id","eligible_windows","sampled_windows"]])

# 2. Freeze a simple participant-disjoint engineering split.
training_people={"S006","S008","S012"}
validation_people={"S013"}
train_idx=np.flatnonzero(metadata["subject_id"].isin(training_people).to_numpy())
val_idx=np.flatnonzero(metadata["subject_id"].isin(validation_people).to_numpy())
assert not(training_people&validation_people)
assert len(X)==len(metadata)==len(y)==128
assert X.shape[1:]==(6,100) and len(train_idx)==96 and len(val_idx)==32
assert set(np.unique(y)).issubset(CLASSES)

# 3. Apply the existing train-only scaler, preserving source row order.
X_train,X_validation,train_statistics=standardise_imu_train_only(
    X[train_idx],X[val_idx]
)
print("Real source Train:",X_train.shape,"| Validation:",X_validation.shape)
print("Actual classes observed in Train:",sorted(np.unique(y[train_idx]).tolist()))
print("All eight classes required in tiny smoke test: NO")

**08. One-Update Training Smoke Test for All Four Architectures**

This smoke test checks that each architecture accepts actual normalised IMU data, computes a finite Cross-Entropy loss, propagates gradients, changes trainable parameters and returns an inference tensor for a different development participant. The **training budget is intentionally one optimiser step**, which cannot meaningfully estimate classifier accuracy or detect Shuffling reliably. All models start with a fixed seed, but their differing parameter counts and dynamics make absolute losses unsuitable as a scientific model ranking.

In [ ]:
# ==================== 08. Real-Source Forward and Backpropagation Smoke Test ====================

# 1. Reuse source class ordering and map only observed Train labels to indices.
mapping={code:i for i,code in enumerate(CLASSES)}
batch_x=torch.from_numpy(X_train[:16])
batch_y=torch.tensor([mapping[int(label)] for label in y[train_idx[:16]]],dtype=torch.long)
val_tensor=torch.from_numpy(X_validation)
training_rows=[]

# 2. Independently initialise and perform one legitimate update per model.
for name in ARCHITECTURES:
    torch.manual_seed(SEED)
    network=build_temporal_model(name,6,8).cpu()
    optimizer=torch.optim.Adam(network.parameters(),lr=1e-3)
    before=[weight.detach().clone() for weight in network.parameters()]
    network.train()
    optimizer.zero_grad(set_to_none=True)
    loss=nn.CrossEntropyLoss()(network(batch_x),batch_y)
    assert torch.isfinite(loss).item()
    loss.backward()
    assert all(
        parameter.grad is not None and torch.isfinite(parameter.grad).all().item()
        for parameter in network.parameters() if parameter.requires_grad
    )
    optimizer.step()
    changed=any(not torch.equal(old,current.detach())
                for old,current in zip(before,network.parameters()))

    # 3. Run held-out-person inference without fitting on Validation.
    network.eval()
    with torch.no_grad():
        validation_logits=network(val_tensor)
    assert changed and validation_logits.shape==(len(val_idx),8)
    training_rows.append({
        "architecture":name,"initial_batch_loss":float(loss.item()),
        "optimizer_updates":1,"parameters_changed":changed,
        "held_out_output_shape":str(tuple(validation_logits.shape)),
        "finite_validation_logits":bool(torch.isfinite(validation_logits).all().item())
    })
smoke=pd.DataFrame(training_rows)
display(smoke.round(4))
print("Smoke training is not a predictive-performance benchmark.")

**09. Model-Design Differences, Hypotheses and Risks for Stage 23**

These architectures do not merely differ in layer names. TCN enlarges receptive fields using dilations; Multi-Scale CNN aggregates several convolution kernel lengths; ResNet1D uses deeper residual feature extraction; Transformer uses global bidirectional token interactions with temporal position encoding. Those differences create testable hypotheses, not evidence of superiority. Fair evaluation needs a common data/sample protocol, subject-held-out splits, a predeclared primary metric and proper class-support reporting.

Class imbalance must also remain visible. Stage 21 verified uneven activity support between people and substantial minority-window loss under chronological caps. Stage 23 must not increase performance by inadvertently changing the source rows or omitting difficult classes.

In [ ]:
# ==================== 09. Architecture-Specific Research Risks and Controls ====================

# 1. Define the intended modelling hypothesis and its primary constraint.
design_map=pd.DataFrame([
    {"architecture":"TCN","hypothesis":"Dilated temporal context helps distinguish related movement sequences","principal_risk":"Noncausal window context and receptive-field assumptions"},
    {"architecture":"Multi-Scale CNN","hypothesis":"Parallel kernels capture multiple movement timescales","principal_risk":"More filters, extra optimisation and potentially noisy small activities"},
    {"architecture":"1D-ResNet","hypothesis":"Residual depth learns expressive sequence features","principal_risk":"Overfitting and stride-related information loss"},
    {"architecture":"Time-Series Transformer","hypothesis":"Bidirectional self-attention captures longer-range within-window patterns","principal_risk":"Computational cost and data hunger"},
])
display(design_map)

# 2. Record the explicit Stage 23 methodological controls.
controls=pd.DataFrame([
    {"constraint":"Same windows and subject folds","status":"Required"},
    {"constraint":"Train-only sensor standardisation and class weights","status":"Required"},
    {"constraint":"Fixed eight-code class denominator and Macro F1","status":"Required"},
    {"constraint":"Per-class Shuffling/stairs performance and support","status":"Required"},
    {"constraint":"Compute/parameters/latency reporting","status":"Required"},
    {"constraint":"Original Test subjects S023/S032 excluded","status":"Required"},
    {"constraint":"Reproducibility over more than one seed","status":"Stage 24"},
])
display(controls)

**10. Scientific Quality Gate and Handover to Matched Model Benchmarking**

The quality gate checks the existence and validity of *all four* advanced architectures, finite logits, parameter updates on genuinely sourced training windows, held-out participant boundaries and absence of original Test subjects. It does not claim full training convergence, class-level predictive performance, calibrated confidence or external generalisation.

**Next stage:** Stage 23 will train and compare the four new models with Random Forest, Standard CNN and Weighted CNN, recording matched source-row identities, split membership, per-class metrics and compute. Larger CPU jobs may be broken into manageable runs, but the final scientific comparison must remain integrated and evidence-complete.

In [ ]:
# ==================== 10. Scientific Quality Gate ====================

# 1. Verify all requested architectures and the shared source interface.
assert len(ARCHITECTURES)==4
assert set(ARCHITECTURES)==set(smoke["architecture"])
assert set(capacity["architecture"])==set(ARCHITECTURES)|{"Simple Temporal CNN"}
assert all(capacity["trainable_parameters"]>0)
assert capacity["logits_shape"].eq("(16, 8)").all()

# 2. Confirm real raw HARTH windows came only from development participants.
assert X.shape==(128,6,100)
assert metadata["subject_id"].nunique()==4
assert not(set(metadata["subject_id"])&RESERVED_TEST)
assert not(training_people&validation_people)
assert np.isfinite(X_train).all() and np.isfinite(X_validation).all()

# 3. Check the four actual one-update training and inference results.
assert smoke["optimizer_updates"].eq(1).all()
assert smoke["parameters_changed"].all()
assert smoke["finite_validation_logits"].all()
assert smoke["held_out_output_shape"].eq("(32, 8)").all()

# 4. Print unambiguous engineering versus scientific completion status.
print("\n"+"="*66)
print("STAGE 22: FOUR ADVANCED TEMPORAL ARCHITECTURES")
print("="*66)
print("Temporal Convolutional Network: IMPLEMENTED AND SMOKE-TESTED")
print("Multi-Scale CNN: IMPLEMENTED AND SMOKE-TESTED")
print("1D-ResNet: IMPLEMENTED AND SMOKE-TESTED")
print("Time-Series Transformer: IMPLEMENTED AND SMOKE-TESTED")
print("Common (batch, channels, time) contract: PASSED")
print("Real-data Train/Validation participant separation: PASSED")
print("Finite gradients and one-step updates: PASSED")
print("Original protected Test used: NO")
print("-"*66)
print("Stage 22 engineering quality gate: PASSED")
print("Full matched model benchmark performed: NO")
print("Independent predictive generalisation established: NO")
print("Next stage: Stage 23 - Unified Model Benchmark")
print("="*66)